## 0. Split Dataset 8:2 - train:test

In [1]:
import zarr
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
import os
import json
from tqdm import tqdm
import matplotlib.pyplot as plt

from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy

DATASET_PATH = "/home/thanh-huynhbao/git/control_policy/dataset/pusht/pusht_cchi_v7_replay.zarr"
OUTPUT_PATH = "/home/thanh-huynhbao/git/control_policy/tutorials/lerobot_implement/outputs/train_diffusion_pusht_local"
CHECKPOINT_DIR = OUTPUT_PATH
CHECKPOINT_NAME = "checkpoint_last.pt"
OUTPUT_PLOT = os.path.join(OUTPUT_PATH, "diffusion_test_inference_eval.png")
# HyperParameters
SEED = 42
N_OBS_STEPS = 2        # Number of historical observations conditioned on (t-1, t)
HORIZON = 16           # Action prediction sequence length (must be a multiple of 8)
N_ACTION_STEPS = 8     # Actions executed per rollout
BATCH_SIZE = 64
NUM_WORKERS = 8
LR = 1e-4
WEIGHT_DECAY = 1e-6
EPOCH = 40
NUM_TRAIN_TIMESTEPS = 100

/home/thanh-huynhbao/git/control_policy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data-driven

In [2]:
class PushTDataset(Dataset):
    def __init__(
        self,
        zarr_path: str,
        episode_indices: list[int],
        horizon: int = 16,
        n_obs_steps: int = 2,
        stats: dict | None = None,
    ):
        self.horizon = horizon
        self.n_obs_steps = n_obs_steps
        self.episode_indices = episode_indices

        # dataset
        store = zarr.open(DATASET_PATH,mode="r")
        all_actions = store["data"]["action"][:]  # (N, 2)
        all_states = store["data"]["state"][:]  # (N, 5)
        all_images = store["data"]["img"][:]  # (N, 96, 96, 3)
        all_episode_ends = store["meta"]["episode_ends"][:] 

        # Build list of episodes: (start_idx, end_idx)
        self.samples = []  # list of (global_t, ep_start, ep_end)
        self.actions = []
        self.states = []
        self.images = []

        for episode_idx in episode_indices:
            episode_start = 0 if episode_idx == 0 else all_episode_ends[episode_idx - 1]
            episode_end = all_episode_ends[episode_idx]
            for t in range(episode_start, episode_end):
                self.samples.append((t, episode_start, episode_end))

        ###############################################
        ## Actions, States and Images from Datasets  ##
        ###############################################
        self.raw_actions = all_actions
        self.raw_states = all_states
        self.raw_images = all_images

        # Compute statistics
        if stats is None:
            # Gather all train set states and actions to compute stats 
            train_states = np.concatenate(
                [
                    all_states[(0 if episode == 0 else all_episode_ends[episode - 1]) : all_episode_ends[episode]] for episode in episode_indices
                ], axis=0,
            )

            train_actions = np.concatenate(
                [
                    all_actions[(0 if episode == 0 else all_episode_ends[episode - 1]) : all_episode_ends[episode]] for episode in episode_indices
                ], axis=0,
            )

            self.stats = {
                "state_mean": train_states.mean(axis=0).tolist(),
                "state_std": np.clip(train_states.std(axis=0), a_min=1e-4, a_max=None).tolist(),
                "action_mean": train_actions.mean(axis=0).tolist(),
                "action_std": np.clip(train_actions.std(axis=0), a_min=1e-4, a_max=None).tolist(),
            }
        else:
            self.stats = stats

        self.state_mean = np.array(self.stats["state_mean"], dtype=np.float32)
        self.state_std = np.array(self.stats["state_std"], dtype=np.float32)
        self.action_mean = np.array(self.stats["action_mean"], dtype=np.float32)
        self.action_std = np.array(self.stats["action_mean"], dtype=np.float32)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        t, episode_start, episode_end = self.samples[idx]

        # 1. Observation sequence: indices [t - n_obs_steps + 1, ..., t]
        obs_indices = [max(episode_start, t - i) for i in reversed(range(self.n_obs_steps))]

        # Stack images: (n_obs_steps, 3, 96, 96)
        raw_imgs = self.raw_images[obs_indices].astype(np.float32) / 255.0
        imgs = np.transpose(raw_imgs, (0, 3, 1, 2))

        # Stack states: (n_obs_steps, 5)
        raw_states = self.raw_states[obs_indices].astype(np.float32)
        states = (raw_states - self.state_mean) / self.state_std

        # 2. Action trajectory: indices starting from (t - n_obs_steps + 1)
        action_start = t - (self.n_obs_steps - 1)
        action_end = action_start + self.horizon

        actions_chunk = np.zeros((self.horizon, 2), dtype=np.float32)
        action_is_pad = np.zeros((self.horizon,), dtype=bool)

        for i, step in enumerate(range(action_start, action_end)):
            if step < episode_start:
                # Clamp before start
                raw_act = self.raw_actions[episode_start].astype(np.float32)
            elif step < episode_end:
                raw_act = self.raw_actions[step].astype(np.float32)
            else:
                # Pad past end
                raw_act = self.raw_actions[episode_end - 1].astype(np.float32)
                action_is_pad[i] = True
            actions_chunk[i] = (raw_act - self.action_mean) / self.action_std

        return {
            "observation.image": torch.from_numpy(imgs),
            "observation.state": torch.from_numpy(states),
            "action": torch.from_numpy(actions_chunk),
            "action_is_pad": torch.from_numpy(action_is_pad),
        }

    
def split_episodes(total_episodes: int, train_ratio: float=0.8, seed: int = 42):
    """Split episode indices into train and test sets."""
    np.random.seed(seed)
    shuffled_indices = np.random.permutation(total_episodes).tolist()
    n_train = int(total_episodes * train_ratio)
    train_indices = sorted(shuffled_indices[:n_train])
    test_indices = sorted(shuffled_indices[n_train:])

    return train_indices, test_indices

In [3]:

# 1. Inspect episodes and split 8:2
store = zarr.open(DATASET_PATH, mode="r")
episode_ends = store["meta"]["episode_ends"][:]
total_episodes = len(episode_ends)
train_indices, test_indices = split_episodes(total_episodes, train_ratio=0.8, seed=SEED)
print(f"Total episodes: {total_episodes}")
print(f"Train episodes: {len(train_indices)} ({len(train_indices)/total_episodes*100:.1f}%)")
print(f"Test episodes: {len(test_indices)} ({len(test_indices)/total_episodes*100:.1f}%)")

# 2. Build datasets
train_dataset = PushTDataset(DATASET_PATH, train_indices, horizon=HORIZON, n_obs_steps=N_OBS_STEPS)
test_dataset = PushTDataset(DATASET_PATH, test_indices, horizon=HORIZON, n_obs_steps=N_OBS_STEPS)
# 2. Data loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f"Train transitions: {len(train_dataset)}, Batches: {len(train_loader)}")
print(f"Test transitions:  {len(test_dataset)}, Batches: {len(test_loader)}") 


Total episodes: 206
Train episodes: 164 (79.6%)
Test episodes: 42 (20.4%)
Train transitions: 20353, Batches: 318
Test transitions:  5297, Batches: 83


## 2. Model Prepare

In [4]:
# Select Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 3. Configure LeRobot Diffusion Policy
config = DiffusionConfig(
    input_features={
        "observation.image": PolicyFeature(type=FeatureType.VISUAL, shape=(3, 96, 96)),
        "observation.state": PolicyFeature(type=FeatureType.STATE, shape=(5,)),
    },
    output_features={
        "action": PolicyFeature(type=FeatureType.ACTION, shape=(2,)),
    },
    n_obs_steps=N_OBS_STEPS,
    horizon=HORIZON,
    n_action_steps=N_ACTION_STEPS,
    vision_backbone="resnet18",
    crop_shape=None,
    crop_ratio=1.0,
    spatial_softmax_num_keypoints=32,
    down_dims=(512, 1024, 2048),
    kernel_size=5,
    n_groups=8,
    diffusion_step_embed_dim=128,
    use_film_scale_modulation=True,
    noise_scheduler_type="DDPM",
    num_train_timesteps=NUM_TRAIN_TIMESTEPS,
    beta_schedule="squaredcos_cap_v2",
    prediction_type="epsilon",
    clip_sample=True,
    clip_sample_range=1.0,
    optimizer_lr=LR,
    optimizer_weight_decay=WEIGHT_DECAY,
)

policy = DiffusionPolicy(config).to(device)

# 4. Optimizer
optimizer = torch.optim.AdamW(
    policy.get_optim_params(),
    lr=LR,
    betas=(0.95, 0.999),
    eps=1e-8,
    weight_decay=WEIGHT_DECAY,
)

# 5. Save metadata
meta_info = {
    "train_indices": train_indices,
    "test_indices": test_indices,
    "horizon": HORIZON,
    "n_obs_steps": N_OBS_STEPS,
    "stats": train_dataset.stats,
}

os.makedirs(OUTPUT_PATH, exist_ok=True)
with open(os.path.join(OUTPUT_PATH, "dataset_meta.json"), "w") as f:
    json.dump(meta_info, f, indent=2)
    print("Meta saved")

Using device: cuda
Meta saved


## 3. Train

In [5]:
print(f"\nStarting Diffusion Policy Training...")

train_loss_list = []

for epoch in range(1, EPOCH + 1):
    policy.train()
    train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCH} [Train]")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        loss, _ = policy(batch)  # DiffusionPolicy returns (loss, None)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), max_norm=10.0)
        optimizer.step()

        train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    train_loss_list.append(avg_train_loss)
    print(f"Epoch: {epoch} - MSE Loss: {avg_train_loss:.4f}")


Starting Diffusion Policy Training...


Epoch 1/40 [Train]: 100%|██████████| 318/318 [01:45<00:00,  3.01it/s, loss=0.0878]


Epoch: 1 - MSE Loss: 0.2078


Epoch 2/40 [Train]:  79%|███████▊  | 250/318 [01:17<00:20,  3.25it/s, loss=0.0597]


KeyboardInterrupt: 

In [ ]:
checkpoint = {
    "epoch": epoch,
    "policy_state_dict": policy.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": config,
    "train_loss": avg_train_loss,
}
torch.save(checkpoint, os.path.join(OUTPUT_PATH, "checkpoint_last.pt"))

## 4. Inference with testset

In [ ]:
EVAL_EPISODE = 1

def load_model_and_metadata(checkpoint_path: str, meta_path: str, device: torch.device):
    """Load trained DiffusionPolicy, configuration, and dataset metadata."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"Metadata not found: {meta_path}")

    with open(meta_path, "r") as f:
        meta_info = json.load(f)

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = checkpoint["config"]

    policy = DiffusionPolicy(config).to(device)
    policy.load_state_dict(checkpoint["policy_state_dict"])
    policy.eval()

    return policy, config, meta_info

def run_episode_inference(
    policy: DiffusionPolicy,
    episode_images: np.ndarray,
    episode_states: np.ndarray,
    stats: dict,
    device: torch.device,
) -> np.ndarray:
    state_min = np.array(stats["state_min"], dtype=np.float32)
    state_max = np.array(stats["state_max"], dtype=np.float32)
    action_min = np.array(stats["action_min"], dtype=np.float32)
    action_max = np.array(stats["action_max"], dtype=np.float32)
    
    state_denom = np.clip(state_max - state_min, a_min=1e-5, a_max=None)
    action_range = action_max - action_min

    T = len(episode_images)
    predicted_actions = []

    # Reset queues (clears observation and action deques)
    policy.reset()

    with torch.no_grad():
        for t in range(T):
            # 1. Normalize current frame
            img = episode_images[t].astype(np.float32) / 255.0
            img = np.transpose(img, (2, 0, 1))
            img_tensor = torch.from_numpy(img).unsqueeze(0).to(device)  # (1, 3, 96, 96)

            norm_state = 2.0 * (episode_states[t].astype(np.float32) - state_min) / state_denom - 1.0
            state_tensor = torch.from_numpy(norm_state).unsqueeze(0).to(device)  # (1, 5)

            batch = {
                "observation.image": img_tensor,
                "observation.state": state_tensor,
            }

            # 2. Select action: internal queue handles history and chunk execution
            pred_norm_action = policy.select_action(batch)
            pred_norm_action = pred_norm_action.cpu().numpy().squeeze(0)

            # 3. Unnormalize [-1, 1] back to raw action space
            pred_action = (pred_norm_action + 1.0) / 2.0 * action_range + action_min
            predicted_actions.append(pred_action)

    return np.array(predicted_actions)

def compute_metrics(ground_truth: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    """Compute MSE, RMSE, MAE, and per-axis errors between GT and predictions."""
    diff = predicted - ground_truth 
    mse = np.mean(diff ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(diff))
    max_err = np.max(np.abs(diff))

    mae_x = np.mean(np.abs(diff[:, 0]))
    mae_y = np.mean(np.abs(diff[:, 1])) 

    return {
        "MSE": float(mse),
        "RMSE": float(rmse),
        "MAE": float(mae),
        "MAE_X": float(mae_x),
        "MAE_Y": float(mae_y),
        "Max_Error": float(max_err),
    } 

def plot_comparison(
    gt_actions: np.ndarray,
    pred_actions: np.ndarray,
    episode_idx: int,
    metrics: dict[str, float],
    output_path: str,
    action_gt: np.ndarray,
    action_pred: np.ndarray
):
    """Plot 2D trajectory and time-series comparison between GT and predicted actions."""
    fig = plt.figure(figsize=(15, 12))

    # Subplot 1: 2D Trajectory
    ax1 = fig.add_subplot(2, 2, 1)
    ax1.plot(gt_actions[:, 0], gt_actions[:, 1], "b-", linewidth=2.0, label="Ground Truth", alpha=0.8)
    ax1.plot(pred_actions[:, 0], pred_actions[:, 1], "r--", linewidth=2.0, label="ACT Prediction", alpha=0.8)

    # Mark Start and End
    ax1.scatter(gt_actions[0, 0], gt_actions[0, 1], c="green", s=90, marker="o", label="Start", zorder=5)
    ax1.scatter(gt_actions[-1, 0], gt_actions[-1, 1], c="blue", s=90, marker="x", label="GT End", zorder=5)
    ax1.scatter(pred_actions[-1, 0], pred_actions[-1, 1], c="red", s=90, marker="^", label="Pred End", zorder=5)

    ax1.set_title(f"Test Episode {episode_idx}: 2D Action Trajectory", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Action Dim 0 (X coordinate)", fontsize=10)
    ax1.set_ylabel("Action Dim 1 (Y coordinate)", fontsize=10)
    ax1.legend(loc="best", fontsize=9)
    ax1.grid(True, linestyle="--", alpha=0.6)

    # Subplot 2: Time Series for X and Y
    ax2 = fig.add_subplot(4, 2, 2)
    ax2.plot(gt_actions[:, 0], "b-", label="GT X", linewidth=1.8)
    ax2.plot(pred_actions[:, 0], "r--", label="Pred X", linewidth=1.8)
    ax2.set_title(f"Action Dim 0 (X) [MAE: {metrics['MAE_X']:.2f}]", fontsize=11, fontweight="bold")
    ax2.set_ylabel("Position X", fontsize=9)
    ax2.legend(loc="upper right", fontsize=8)
    ax2.grid(True, linestyle="--", alpha=0.6)

    ax3 = fig.add_subplot(4, 2, 4)
    ax3.plot(gt_actions[:, 1], "b-", label="GT Y", linewidth=1.8)
    ax3.plot(pred_actions[:, 1], "r--", label="Pred Y", linewidth=1.8)
    ax3.set_title(f"Action Dim 1 (Y) [MAE: {metrics['MAE_Y']:.2f}]", fontsize=11, fontweight="bold")
    ax3.set_xlabel("Time Step", fontsize=10)
    ax3.set_ylabel("Position Y", fontsize=9)
    ax3.legend(loc="upper right", fontsize=8)
    ax3.grid(True, linestyle="--", alpha=0.6)

    # Subplot 2: Time Series for X and Y
    ax4 = fig.add_subplot(4, 1, 3)
    ax4.plot(action_gt[:, 0], "b-", label="GT X", linewidth=1.8)
    ax4.plot(action_pred[:, 0], "r--", label="Pred X", linewidth=1.8)
    ax4.set_title(f"Action Dim 0 (X) [MAE: {metrics['MAE_X']:.2f}]", fontsize=11, fontweight="bold")
    ax4.set_ylabel("Position X", fontsize=9)
    ax4.legend(loc="upper right", fontsize=8)
    ax4.grid(True, linestyle="--", alpha=0.6)

    ax5 = fig.add_subplot(4, 1, 4)
    ax5.plot(action_gt[:, 1], "b-", label="GT Y", linewidth=1.8)
    ax5.plot(action_pred[:, 1], "r--", label="Pred Y", linewidth=1.8)
    ax5.set_title(f"Action Dim 1 (Y) [MAE: {metrics['MAE_Y']:.2f}]", fontsize=11, fontweight="bold")
    ax5.set_xlabel("Time Step", fontsize=10)
    ax5.set_ylabel("Position Y", fontsize=9)
    ax5.legend(loc="upper right", fontsize=8)
    ax5.grid(True, linestyle="--", alpha=0.6)

    plt.suptitle(
        f"ACT Push-T Evaluation (Episode {episode_idx}) | Overall MAE: {metrics['MAE']:.2f}, RMSE: {metrics['RMSE']:.2f}",
        fontsize=13,
        fontweight="bold",
        y=0.98,
    )
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved inference visualization plot to: {output_path}")


In [ ]:
# 1. Load model and metadata
checkpoint_path = os.path.join(CHECKPOINT_DIR, CHECKPOINT_NAME)
meta_path = os.path.join(CHECKPOINT_DIR, "dataset_meta.json")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

policy, config, meta_info = load_model_and_metadata(checkpoint_path, meta_path, device)
test_indices = meta_info["test_indices"]
stats = meta_info["stats"]

print(f"Loaded checkpoint: {checkpoint_path}")
print(f"Test episodes count: {len(test_indices)} (indices: {test_indices[:5]} ...)")

# 2. Open dataset 
store = zarr.open(DATASET_PATH, mode="r")
actions = store["data"]["action"][:]
states = store["data"]["state"][:]
images = store["data"]["img"][:]

episode_ends = store["meta"]["episode_ends"][:]

# 3. Evaluate test set
all_metrics = []
plot_gt, plot_pred, plot_metrics = None, None, None
target_plot_ep = EVAL_EPISODE if EVAL_EPISODE is not None else test_indices[0]
action_gt = np.empty((0,2))
action_pred = np.empty((0,2))
print(f"Running inference!")
for episode_idx in tqdm(test_indices, desc="Evaluating Test Episodes"):
    episode_start = 0 if episode_idx == 0 else episode_ends[episode_idx - 1]
    episode_end = episode_ends[episode_idx]

    episode_gt_actions = actions[episode_start:episode_end]
    episode_states = states[episode_start:episode_end]
    episode_images = images[episode_start:episode_end]

    action_gt = np.vstack([action_gt, episode_gt_actions])

    episode_predicted_actions = run_episode_inference(policy, episode_images, episode_states, stats, device)
    episode_metrics = compute_metrics(episode_gt_actions, episode_predicted_actions)
    all_metrics.append(episode_metrics) 

    action_pred = np.vstack([action_pred, episode_predicted_actions])

    if episode_idx == target_plot_ep:
        plot_gt = episode_gt_actions 
        plot_pred = episode_predicted_actions 
        plot_metrics = episode_metrics 
print(f"Action GT type: {type(action_gt)}")
print(f"Action pred type: {type(action_pred)}")

print(f"Action GT type: {action_gt.shape}")
print(f"Action pred type: {action_pred.shape}")

print(f"Action GT type: {episode_gt_actions.shape}")
print(f"Action pred type: {episode_predicted_actions.shape}")

# # 5. plot for target_episode
# if plot_gt is not None and plot_pred is not None:
#     plot_comparison(plot_gt, plot_pred, target_plot_ep, plot_metrics, OUTPUT_PLOT, action_gt, action_pred)

## 5. Plot and evaluate

In [ ]:
if plot_gt is not None and plot_pred is not None:
    plot_comparison(plot_gt, plot_pred, target_plot_ep, plot_metrics, OUTPUT_PLOT, action_gt, action_pred)

In [ ]:
# 4. Aggregate metrics 
mean_mse = np.mean([m["MSE"] for m in all_metrics])
mean_rmse = np.mean([m["RMSE"] for m in all_metrics])
mean_mae = np.mean([m["MAE"] for m in all_metrics])
mean_mae_x = np.mean([m["MAE_X"] for m in all_metrics])
mean_mae_y = np.mean([m["MAE_Y"] for m in all_metrics])
max_err = np.max([m["Max_Error"] for m in all_metrics])
print("\n" + "=" * 55)
print("           TEST SET EVALUATION METRICS SUMMARY           ")
print("=" * 55)
print(f"Evaluated Test Episodes: {len(test_indices)}")
print(f"Mean Squared Error (MSE):      {mean_mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {mean_rmse:.4f}")
print(f"Mean Absolute Error (MAE):     {mean_mae:.4f}")
print(f"  - Action X MAE:              {mean_mae_x:.4f}")
print(f"  - Action Y MAE:              {mean_mae_y:.4f}")
print(f"Max Absolute Error:            {max_err:.4f}")
print("=" * 55)

In [6]:
import gc
import torch

def clear_gpu_memory():
    print(f"Allocated before: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached before:    {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    
    gc.collect()
    torch.cuda.empty_cache()
    
    print("--- Memory Cleared ---")
    print(f"Allocated after:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Cached after:     {torch.cuda.memory_reserved() / 1e9:.2f} GB")

# Call the function
clear_gpu_memory()

Allocated before: 4.26 GB
Cached before:    5.69 GB
--- Memory Cleared ---
Allocated after:  4.26 GB
Cached after:     4.76 GB
